In [1]:
import pandas as pd
from pathlib import Path

source = Path("../results/cohort_features_with_glycated_albumin_checkpoint.pkl")
master = pd.read_pickle(source).copy()

assert len(master) == 9957
assert master["SEQN"].is_unique
assert "death_5yr" in master.columns

print("NHANES MASTER DATASET")
print("Participants:", len(master))
print("Total columns:", master.shape[1])
print("Deaths:", int(master["death_5yr"].sum()))
print("Survivors:", int((master["death_5yr"] == 0).sum()))

for col in ["SSGALB", "DERIVED_HL_MED", "DERIVED_HTN_MED", "DERIVED_DM_MED"]:
    print(col, ":", "Present" if col in master.columns else "Missing")

output = Path("../results/nhanes_master_v1.pkl")
master.to_pickle(output)
print("Saved:", output)


NHANES MASTER DATASET
Participants: 9957
Total columns: 300
Deaths: 1293
Survivors: 8664
SSGALB : Present
DERIVED_HL_MED : Present
DERIVED_HTN_MED : Present
DERIVED_DM_MED : Present
Saved: ../results/nhanes_master_v1.pkl


In [2]:
# Identify outcome and administrative columns
blocked_names = {
    "SEQN", "DEATH_5YR", "MORTSTAT",
    "ELIGSTAT", "PERMTH_INT", "PERMTH_EXM"
}

blocked = sorted(
    col for col in master.columns
    if col.upper() in blocked_names
    or col.upper().startswith(("WT", "SDMV", "PERMTH"))
)

print("Columns requiring exclusion from predictors:")
print(blocked)

print("\nNumber flagged:", len(blocked))
print("Other columns:", master.shape[1] - len(blocked))

Columns requiring exclusion from predictors:
['ELIGSTAT', 'MORTSTAT', 'PERMTH_INT', 'SDMVPSU', 'SDMVSTRA', 'SEQN', 'WTINT2YR', 'WTINT4YR', 'WTIREP01', 'WTIREP02', 'WTIREP03', 'WTIREP04', 'WTIREP05', 'WTIREP06', 'WTIREP07', 'WTIREP08', 'WTIREP09', 'WTIREP10', 'WTIREP11', 'WTIREP12', 'WTIREP13', 'WTIREP14', 'WTIREP15', 'WTIREP16', 'WTIREP17', 'WTIREP18', 'WTIREP19', 'WTIREP20', 'WTIREP21', 'WTIREP22', 'WTIREP23', 'WTIREP24', 'WTIREP25', 'WTIREP26', 'WTIREP27', 'WTIREP28', 'WTIREP29', 'WTIREP30', 'WTIREP31', 'WTIREP32', 'WTIREP33', 'WTIREP34', 'WTIREP35', 'WTIREP36', 'WTIREP37', 'WTIREP38', 'WTIREP39', 'WTIREP40', 'WTIREP41', 'WTIREP42', 'WTIREP43', 'WTIREP44', 'WTIREP45', 'WTIREP46', 'WTIREP47', 'WTIREP48', 'WTIREP49', 'WTIREP50', 'WTIREP51', 'WTIREP52', 'WTMEC2YR', 'WTMEC4YR', 'WTMREP01', 'WTMREP02', 'WTMREP03', 'WTMREP04', 'WTMREP05', 'WTMREP06', 'WTMREP07', 'WTMREP08', 'WTMREP09', 'WTMREP10', 'WTMREP11', 'WTMREP12', 'WTMREP13', 'WTMREP14', 'WTMREP15', 'WTMREP16', 'WTMREP17', 'WTMREP18

In [3]:
# Remove blocked, empty, and constant columns
constant_cols = [
    c for c in master.columns
    if master[c].nunique(dropna=True) <= 1
]

excluded = set(blocked) | set(constant_cols) | {"RIDSTATR"}

X_expanded = master.drop(columns=list(excluded)).copy()
y_expanded = master["death_5yr"].astype(int).copy()

# Safety checks
assert len(X_expanded) == 9957
assert "death_5yr" not in X_expanded.columns
assert "MORTSTAT" not in X_expanded.columns
assert "SEQN" not in X_expanded.columns
assert X_expanded.columns.is_unique

print("EXPANDED DATASET")
print("Participants:", len(X_expanded))
print("Candidate predictors:", X_expanded.shape[1])
print("Empty or constant columns excluded:", len(constant_cols))
print("Missing-value cells:", int(X_expanded.isna().sum().sum()))
print("Deaths:", int(y_expanded.sum()))

# Save separately, without overwriting the master
X_expanded.to_pickle("../results/expanded_candidates_v1.pkl")

print("\nCandidate dataset saved successfully.")

EXPANDED DATASET
Participants: 9957
Candidate predictors: 180
Empty or constant columns excluded: 5
Missing-value cells: 673246
Deaths: 1293

Candidate dataset saved successfully.


In [4]:
import pandas as pd

review = pd.DataFrame({
    "Variable": X_expanded.columns,
    "Missing_%": (X_expanded.isna().mean() * 100).round(1).values
})

print("Candidate predictors:", len(review))
print("70% or more missing:",
      (review["Missing_%"] >= 70).sum())

print("\nHigh-missingness variables:")
display(
    review[review["Missing_%"] >= 70]
    .sort_values("Missing_%", ascending=False)
)

print("\nAdministrative columns requiring review:")
print([
    c for c in X_expanded.columns
    if c.upper() in {"RX_USE", "RIDSTATR", "MORTSTAT", "ELIGSTAT"}
    or c.upper().startswith(("HAS_", "PERMTH"))
])

Candidate predictors: 180
70% or more missing: 58

High-missingness variables:


,Variable,Missing_%
85,CVDR3HR,95.3
68,SS3ADLG,94.1
65,SSTESTO,94.0
66,SSSHBG,94.0
67,SSSE2,94.0
86,CVDESVO2,90.9
87,CVDFITLV,90.9
82,CVDWHR,90.7
83,CVDR1HR,90.5
84,CVDR2HR,90.0



Administrative columns requiring review:
['has_BIX', 'RX_USE']


In [5]:
from pathlib import Path

# Exclude unverified context variables
review_columns = ["has_BIX", "RX_USE"]

X_candidates_v2 = X_expanded.drop(
    columns=review_columns,
    errors="ignore"
).copy()

assert len(X_candidates_v2) == 9957
assert "death_5yr" not in X_candidates_v2.columns
assert "SEQN" not in X_candidates_v2.columns

output = Path("../results/expanded_candidates_v2.pkl")
X_candidates_v2.to_pickle(output)

print("Participants:", len(X_candidates_v2))
print("Candidate predictors:", X_candidates_v2.shape[1])
print("Saved:", output)

Participants: 9957
Candidate predictors: 178
Saved: ../results/expanded_candidates_v2.pkl
